# Hafta 10 · Kuantum Fourier Dönüşümü, Faz Kestirimi ve Shor Algoritması
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta üç halkalı bir zincir kuruyoruz: **QFT** (genlik vektörüne FFT) → **faz kestirimi / QPE** (gizli bir açıyı bitlerle okumak) → **Shor** (çarpanlara ayırmayı periyot bulmaya indirgemek). Her adımı önce NumPy ile, sonra Qiskit devresiyle yapıp sonuçları karşılaştıracağız.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Klasik DFT/FFT hatırlatma: `np.fft`, periyodik dizinin spektrumu | 6 dk |
| B | QFT = genlik vektörüne DFT; `np.fft` ile normalizasyon/işaret farkı | 7 dk |
| C | QFT devresini elle kurmak (H + kontrollü faz + SWAP), `QFTGate`, Bloch'ta saat ibreleri | 9 dk |
| D | Faz kestirimi (QPE): T, S ve keyfi faz; hassasiyet tablosu | 10 dk |
| E | Shor'un klasik kısmı: periyot, gcd, sürekli kesirler | 6 dk |
| F | Shor'un kuantum kısmı: N = 15, a = 7 uçtan uca | 9 dk |
| G | Alıştırmalar (8 adet, `assert` ile kontrol) | ödev |

**Bit sırası:** Dersin tamamında olduğu gibi **Qiskit sırası** (q₀ en sağda): `|x⟩` durumunun indeksi x tamsayısıdır.

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
aer = AerSimulator(seed_simulator=2026)

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (1. haftadaki fonksiyon)."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
        u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
        ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                        color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
        t = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
        for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
        for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
            ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
        x, y, z = state_to_bloch(amps)
        ax.plot([0, x], [0, y], [0, z], color=BLUE, lw=3); ax.scatter([x], [y], [z], color=BLUE, s=60, depthshade=False)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()

def run_counts(qc, shots=2048):
    """Devreyi Aer'de çalıştırıp sayımları döndürür."""
    return aer.run(transpile(qc, aer), shots=shots).result().get_counts()
print("hazır")

---
## A · Klasik DFT/FFT hatırlatma
Ayrık Fourier dönüşümü (DFT), N elemanlı bir diziyi **frekans bileşenlerine** ayırır. Ses işleme, görüntü sıkıştırma (JPEG), polinom çarpımı... hepsinde kullanılır. Naif hesabı N² işlemdir; **FFT** aynı sonucu N·log₂N işlemde verir.

$$\text{np.fft.fft: } \; X_k = \sum_{x=0}^{N-1} a_x \, e^{-2\pi i\, x k / N}$$

**Bu haftanın kilit gözlemi:** Periyodu r olan bir dizinin spektrumunda tepeler **N/r'nin katlarında** çıkar. Tepelerin aralığından periyodu okuyabiliriz. Shor algoritması tam olarak bunu yapacak.

In [ ]:
# İki sinüsün toplamı: FFT hangi frekansları içerdiğini söyler
Ns = 128; t = np.arange(Ns)
sig = np.sin(2*np.pi*5*t/Ns) + 0.6*np.sin(2*np.pi*12*t/Ns)
S = np.abs(np.fft.fft(sig))[:Ns//2]
print("En büyük iki bileşen:", np.argsort(S)[-2:][::-1])

fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(t, sig, color=BLUE); axs[0].set_title("zaman alanı")
axs[1].bar(range(Ns//2), S, color=NAVY); axs[1].set_title("|FFT|: frekans alanı")
plt.show()

In [ ]:
# Periyodik dizi: f[x] = 1 eğer x mod 4 == 1  (periyot r = 4), N = 16
N = 16
f = np.array([1.0 if x % 4 == 1 else 0.0 for x in range(N)])
F = np.fft.fft(f)
print("f      =", f)
print("|F|    =", np.abs(F).round(3))
print("tepeler:", np.nonzero(np.abs(F) > 1e-9)[0], " -> aralık N/r =", N // 4)

# Kendi DFT'miz (naif, O(N²)) ile karşılaştırma
def dft(a):
    N = len(a); x = np.arange(N)
    W = np.exp(-2j*np.pi*np.outer(x, x)/N)
    return W @ a
print("naif DFT == np.fft.fft ?", np.allclose(dft(f), F))

---
## B · QFT = genlik vektörüne DFT
Kuantum Fourier dönüşümü (QFT), 2ⁿ elemanlı **durum vektörüne** DFT uygular. Tek farklar:
- İşaret **+** (np.fft.fft'te −),
- Ölçek **1/√N** (üniter olsun diye; olasılık toplamı 1 kalmalı).

$$\text{QFT}|x\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{+2\pi i\, x k / N} |k\rangle \qquad\Rightarrow\qquad \text{QFT}(a) = \sqrt{N}\cdot \texttt{np.fft.ifft}(a) = \texttt{np.fft.ifft(a, norm="ortho")}$$

In [ ]:
def qft_matrix(n):
    N = 2**n; j = np.arange(N)
    return np.exp(2j*np.pi*np.outer(j, j)/N) / np.sqrt(N)

n = 3; N = 2**n
F8 = qft_matrix(n)
print("Üniter mi?", np.allclose(F8.conj().T @ F8, np.eye(N)))

a = np.random.default_rng(1).normal(size=N) + 1j*np.random.default_rng(2).normal(size=N)
a /= np.linalg.norm(a)
print("QFT(a) == sqrt(N)*ifft(a)      ?", np.allclose(F8 @ a, np.sqrt(N)*np.fft.ifft(a)))
print("QFT(a) == ifft(a, norm='ortho') ?", np.allclose(F8 @ a, np.fft.ifft(a, norm="ortho")))
print("QFT(a) == fft(a)  (YANLIŞ)      ?", np.allclose(F8 @ a, np.fft.fft(a)))
print("Qiskit QFTGate(3) matrisi aynı mı?", np.allclose(Operator(QFTGate(3)).data, F8))

In [ ]:
# QFT'nin bazal durum üzerindeki etkisi: tüm genliklerin büyüklüğü aynı, fazlar farklı
psi = F8 @ np.eye(N)[5]          # QFT|5⟩
print("genlikler  :", psi.round(3))
print("|genlik|²  :", (abs(psi)**2).round(3), " -> düz dağılım, ölçüm hiçbir şey söylemez!")
print("fazlar (tur):", (np.angle(psi)/(2*np.pi) % 1).round(3), " -> her adımda 5/8 tur")

---
## C · QFT devresi: H + kontrollü faz + SWAP
QFT'nin sihri **çarpan (tensör çarpımı) gösterimidir**: QFT|x⟩ dolanık değildir, her kübit ayrı bir durumdadır:

$$\text{QFT}|x\rangle = \bigotimes_{j=n-1}^{0} \frac{|0\rangle + e^{2\pi i\, x\, 2^j / N}|1\rangle}{\sqrt 2}$$

Yani **kübit qⱼ**, Bloch küresinin ekvatorunda **x·2ʲ/N tur** açısında duran bir **saat ibresidir**: q₀ en yavaş ibre (her x için 1/N tur), en yüksek kübit en hızlı ibre (her x için yarım tur).

Devre: her kübite H (ibreyi ekvatora koy), sonra daha düşük kübitlerden kontrollü faz `cp(π/2^m)` (ince ayar), sonda bit sırasını ters çevirmek için SWAP.

In [ ]:
def qft_circuit(n, swaps=True):
    qc = QuantumCircuit(n, name="QFT")
    for j in reversed(range(n)):             # en yüksek kübitten başla
        qc.h(j)
        for k in reversed(range(j)):          # daha düşük kübitlerden kontrollü faz
            qc.cp(np.pi / 2**(j - k), k, j)
    if swaps:
        for i in range(n // 2):
            qc.swap(i, n - 1 - i)
    return qc

qc3 = qft_circuit(3)
display(qc3.draw("mpl"))
for n in [2, 3, 4, 5]:
    print(f"n={n}: elle kurulan == QFTGate ?", Operator(qft_circuit(n)).equiv(Operator(QFTGate(n))),
          "| kapılar:", dict(qft_circuit(n).count_ops()))

In [ ]:
# Kapı sayısı: n(n+1)/2 + floor(n/2)  ->  O(n²).  Klasik FFT: O(n·2ⁿ)
for n in [4, 8, 16, 32]:
    q = n*(n+1)//2 + n//2
    print(f"n={n:2d}: QFT kapı sayısı = {q:4d}    FFT işlem ~ n·2ⁿ = {n*2**n:.2e}")
print("\nAma dikkat: QFT sonucunu (2ⁿ genliği) OKUYAMAYIZ; ölçüm tek bir k verir.")

In [ ]:
# Qiskit 2.x kütüphane kullanımı: QFTGate (eski QFT sınıfı 2.1'den beri kullanımdan kalkıyor)
qc = QuantumCircuit(3)
qc.x([0, 2])                    # |101⟩ = |5⟩
qc.append(QFTGate(3), [0, 1, 2])
sv = Statevector(qc)
print("QFTGate ile QFT|5⟩ fazları (tur):", (np.angle(sv.data)/(2*np.pi) % 1).round(3))

def qubit_amps(sv_data, j, base=0):
    """Çarpım durumunda kübit j'nin 2'li vektörü: diğer kübitler 'base' indeksinde sabitken
    base ve base + 2^j indekslerindeki genliklerden okunur."""
    v = np.array([sv_data[base], sv_data[base + 2**j]]); return v / np.linalg.norm(v)

amps = [qubit_amps(sv.data, j) for j in [2, 1, 0]]
print("Beklenen açılar (derece):", [(360*5*2**j/8) % 360 for j in [2, 1, 0]])
plot_bloch(amps, ["q2: 180°", "q1: 90°", "q0: 225°"])

---
## D · Faz kestirimi (QPE): gizli açıyı bitlerle okumak
**Problem:** Elimizde bir kapı U ve onun bir **özvektörü** |u⟩ var: U|u⟩ = e^{2πiφ}|u⟩. Yani U bu vektörü değiştirmez, sadece bir faz ekler (2. haftadaki ⟨Z⟩'yi hatırlayın: |0⟩ ve |1⟩, Z'nin özvektörleriydi). **φ'yi (0 ≤ φ < 1) ikili kesir olarak okumak istiyoruz.**

**Fikir:**
1. t sayım kübitine H → hepsi ekvatorda.
2. Sayım kübiti k, U'yu **2ᵏ kez** kontrol eder. Faz kontrol kübitine "geri teper" (**phase kickback**): kübit k'nın ibresi **2ᵏ·φ tur** döner.
3. Bu, tam olarak QFT|y⟩'nin saat ibresi deseni! → **QFT†** uygula → ölçüm y verir, **φ ≈ y / 2ᵗ**.

Örnek: T kapısı |1⟩'e e^{iπ/4} = e^{2πi·(1/8)} ekler → φ = 1/8 = 0.001₂.

In [ ]:
def qpe_circuit(lam, t):
    """P(lam) kapısının |1⟩ özvektörü için QPE. Gerçek faz φ = lam / 2π."""
    qc = QuantumCircuit(t + 1, t)
    qc.x(t)                                  # hedef kübit = özvektör |1⟩
    qc.h(range(t))
    for k in range(t):
        qc.cp(lam * 2**k, k, t)              # kontrollü-U^(2^k)
    qc.append(QFTGate(t).inverse(), range(t))
    qc.measure(range(t), range(t))
    return qc

qc = qpe_circuit(np.pi/4, 3)               # T kapısı
display(qc.draw("mpl"))
c = run_counts(qc)
print("T kapısı, t=3:", c, " -> y = 001 -> φ = 1/8")
print("S kapısı, t=3:", run_counts(qpe_circuit(np.pi/2, 3)), " -> y = 010 -> φ = 2/8 = 1/4")

In [ ]:
# Kickback'i gözle görmek: QFT† öncesi sayım kübitleri (T kapısı, t=3)
qc = QuantumCircuit(4); qc.x(3); qc.h(range(3))
for k in range(3): qc.cp(np.pi/4 * 2**k, k, 3)
sv = Statevector(qc).data
amps = [qubit_amps(sv, j, base=8) for j in [2, 1, 0]]     # hedef q3 = 1 -> taban indeks 8
plot_bloch(amps, ["q2: 4·45° = 180°", "q1: 2·45° = 90°", "q0: 45°"])

### Tam temsil edilemeyen faz: φ = 1/3
1/3 = 0.010101...₂ sonsuz ikili açılıma sahiptir. t bitle tam yazılamaz → ölçüm bir **olasılık dağılımı** verir; tepe, φ'ye en yakın y/2ᵗ değerindedir.

In [ ]:
for t in [3, 6]:
    c = run_counts(qpe_circuit(2*np.pi/3, t), shots=4000)
    ys = np.array([int(k, 2) for k in c]); vals = np.array(list(c.values()))
    best = ys[np.argmax(vals)]
    print(f"t={t}: en sık y = {best} -> φ ≈ {best}/{2**t} = {best/2**t:.4f}  (gerçek 0.3333), P ≈ {vals.max()/4000:.2f}")
    plt.figure(figsize=(9, 2.6)); plt.bar(ys, vals/4000, color=BLUE, width=0.7)
    plt.title(f"QPE, φ = 1/3, t = {t}"); plt.xlabel("y"); plt.show()

In [ ]:
# Hassasiyet tablosu: t sayım kübiti -> çözünürlük 1/2^t
def qpe_probs(t, phi):
    Nn = 2**t; y = np.arange(Nn)
    return np.abs(np.exp(2j*np.pi*np.outer(phi - y/Nn, np.arange(Nn))).sum(1)/Nn)**2

print(" t | çözünürlük 1/2^t | en olası φ̃ | hata      | P(en olası)")
for t in range(2, 11):
    p = qpe_probs(t, 1/3); yb = np.argmax(p)
    print(f"{t:2d} | {1/2**t:16.6f} | {yb/2**t:10.6f} | {abs(yb/2**t-1/3):.6f}  | {p[yb]:.3f}")

---
## E · Shor'un klasik kısmı
**Çarpanlara ayırma → periyot bulma:** N'yi çarpanlarına ayırmak için rastgele bir a seçeriz (gcd(a, N) = 1) ve
$$f(x) = a^x \bmod N$$
fonksiyonunun **periyodunu** r buluruz (a^r ≡ 1 mod N olan en küçük r > 0). r çift ve a^{r/2} ≢ −1 (mod N) ise:
$$\gcd(a^{r/2}-1,\,N) \text{ ve } \gcd(a^{r/2}+1,\,N) \text{ N'nin gerçek çarpanlarıdır.}$$
Klasik olarak r'yi bulmak büyük N için çok yavaştır; **kuantum kısım sadece r'yi bulur**.

In [ ]:
def find_period_classical(a, N):
    """Kaba kuvvet: a^r mod N = 1 olan en küçük r (sadece küçük N için!)."""
    r, v = 1, a % N
    while v != 1:
        v = (v * a) % N; r += 1
    return r

N = 15
print(" a | a^x mod 15 (x=0..7)          | r | a^(r/2) mod 15 | gcd(-1) | gcd(+1)")
for a in [2, 4, 7, 8, 11, 13, 14]:
    r = find_period_classical(a, N); h = pow(a, r//2, N)
    seq = [pow(a, x, N) for x in range(8)]
    print(f"{a:2d} | {str(seq):28s} | {r} | {h:14d} | {gcd(h-1, N):7d} | {gcd(h+1, N):7d}")
print("\n14 için a^(r/2) = 14 ≡ -1 (mod 15): gcd'ler 1 ve 15 -> BAŞARISIZ, yeni a seç.")
print("pow(a, x, N) Python'da hızlı modüler üstür (kare al-çarp yöntemi).")

In [ ]:
# Sürekli kesirler: ölçülen y/2^t -> paydası N'den küçük en yakın kesir s/r
for y in [64, 128, 192, 0, 171]:
    fr = Fraction(y, 256).limit_denominator(15)
    print(f"y = {y:3d}: y/256 = {y/256:.5f}  ->  {fr}  -> r adayı = {fr.denominator}")
print("\nDikkat: y = 128 -> 1/2 -> r adayı 2 (7² mod 15 = 4 ≠ 1): bu durumda r'nin KATI aranır ya da tekrar ölçülür.")

---
## F · Shor'un kuantum kısmı: N = 15, a = 7
QPE'yi **modüler çarpma** üniterine uygularız: U_a|y⟩ = |a·y mod N⟩. Bu bir **permütasyon matrisidir** (16×16, sadece 0 ve 1). |1⟩ durumu U'nun özvektörlerinin eşit karışımıdır ve özdeğer fazları **s/r** biçimindedir → QPE bize s/r'yi verir.

- 8 sayım kübiti (t = 8, 2ᵗ = 256), 4 hedef kübit (0..15 sayıları için)
- Sayım kübiti k, U_a^{2ᵏ} = U_{a^{2ᵏ} mod N} uygular (modüler üs = tekrarlı kare alma)

⚠️ Burada U'yu **matris olarak** veriyoruz (`UnitaryGate`), bu yüzden bu bir *gösterimdir*: gerçek bir Shor uygulaması modüler çarpmayı toplayıcı devrelerinden kurar ve binlerce kapı gerektirir.

In [ ]:
def mult_mod_gate(a, N=15, n=4):
    """|y⟩ -> |a·y mod N⟩ (y < N), y >= N için değişmez. 2^n x 2^n permütasyon."""
    M = np.zeros((2**n, 2**n))
    for y in range(2**n):
        M[(a * y) % N if y < N else y, y] = 1
    return UnitaryGate(M, label=f"×{a} mod {N}")

U7 = mult_mod_gate(7)
v = np.zeros(16); v[1] = 1
path = [1]
for _ in range(4):
    v = Operator(U7).data.real @ v; path.append(int(np.argmax(v)))
print("|1⟩ üzerinde tekrar tekrar ×7:", path, " -> döngü uzunluğu 4 = r")

In [ ]:
def shor_circuit(a, N=15, t=8, n=4):
    qc = QuantumCircuit(t + n, t)
    qc.h(range(t))
    qc.x(t)                                    # hedef register = |1⟩ (q_t en düşük bit)
    for k in range(t):
        qc.append(mult_mod_gate(pow(a, 2**k, N), N, n).control(1), [k] + list(range(t, t + n)))
    qc.append(QFTGate(t).inverse(), range(t))
    qc.measure(range(t), range(t))
    return qc

qc = shor_circuit(7)
display(qc.draw("mpl", fold=-1, scale=0.6))
counts = run_counts(qc, shots=2048)
ys = {int(k, 2): v for k, v in counts.items()}
print("ölçülen y değerleri:", dict(sorted(ys.items())))
plt.figure(figsize=(9, 2.6)); plt.bar(list(ys), list(ys.values()), width=3, color=BLUE)
plt.xticks([0, 64, 128, 192, 255]); plt.title("Shor N=15, a=7"); plt.xlabel("y"); plt.show()

In [ ]:
# Ölçümlerden r'yi çıkar ve çarpanları bul (uçtan uca)
def r_candidates(y, t, N):
    return Fraction(y, 2**t).limit_denominator(N).denominator

a, N = 7, 15
for y in sorted(ys):
    r = r_candidates(y, 8, N)
    ok = pow(a, r, N) == 1
    msg = ""
    if ok and r % 2 == 0 and pow(a, r//2, N) != N - 1:
        h = pow(a, r//2, N); msg = f"çarpanlar: {gcd(h-1, N)} × {gcd(h+1, N)}"
    print(f"y={y:3d} -> {Fraction(y, 256)} -> r adayı {r}: a^r mod N = 1 ? {ok}  {msg}")

---
## G · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · QFT matrisini kendin kur
`my_qft_matrix(n)` fonksiyonu 2ⁿ×2ⁿ QFT matrisini döndürsün (eleman (j, k) = ω^{jk}/√N, ω = e^{2πi/N}). Qiskit `QFTGate` ve `np.fft.ifft(..., norm="ortho")` ile karşılaştırılacak.

In [ ]:
def my_qft_matrix(n):
    # TODO
    pass

for n in [1, 2, 3, 4]:
    assert np.allclose(my_qft_matrix(n), Operator(QFTGate(n)).data)
v = np.arange(8) + 1j; v = v / np.linalg.norm(v)
assert np.allclose(my_qft_matrix(3) @ v, np.fft.ifft(v, norm="ortho"))
print("Alıştırma 1 ✓")

### Alıştırma 2 · SWAP'sız QFT
`qft_no_swap(n)`: QFT devresini **SWAP kapıları olmadan** kurun. Sonra SWAP'ları sona ekleyince `QFTGate(n)` ile eşit olduğunu gösterin. (İpucu: `qft_circuit(n, swaps=False)` yerine kendiniz yazın.)

In [ ]:
def qft_no_swap(n):
    qc = QuantumCircuit(n)
    # TODO: H ve cp kapıları
    return qc

for n in [2, 3, 4]:
    qc = qft_no_swap(n)
    assert "swap" not in qc.count_ops()
    for i in range(n // 2): qc.swap(i, n - 1 - i)
    assert Operator(qc).equiv(Operator(QFTGate(n)))
print("Alıştırma 2 ✓")

### Alıştırma 3 · Saat ibresi açıları
`qft_phases(x, n)`: QFT|x⟩ durumunda kübit q₀, q₁, …, q₍ₙ₋₁₎'in ibre açılarını **tur cinsinden** ([0, 1) aralığında) liste olarak döndürsün. Qiskit durum vektörü ile karşılaştırılacak.

In [ ]:
def qft_phases(x, n):
    # TODO
    pass

for n in [3, 4]:
    for x in range(2**n):
        sv = Statevector.from_int(x, 2**n).evolve(QFTGate(n)).data
        got = qft_phases(x, n)
        for j in range(n):
            amps = qubit_amps(sv, j)
            meas = (np.angle(amps[1] / amps[0]) / (2*np.pi)) % 1
            assert np.isclose(min(abs(meas - got[j]), 1 - abs(meas - got[j])), 0, atol=1e-9)
assert np.allclose(qft_phases(5, 3), [5/8, 2/8, 4/8])
print("Alıştırma 3 ✓")

### Alıştırma 4 · Faz kestirici
`estimate_phase(lam, t, shots)`: `qpe_circuit(lam, t)` devresini çalıştırıp **en sık** ölçülen y'den φ̃ = y/2ᵗ döndürsün. T (λ = π/4), S (λ = π/2) ve keyfi λ = 2π·0.3 için test edilecek.

In [ ]:
def estimate_phase(lam, t, shots=2000):
    # TODO
    pass

assert estimate_phase(np.pi/4, 3) == 0.125
assert estimate_phase(np.pi/2, 4) == 0.25
est = estimate_phase(2*np.pi*0.3, 7)
print("φ = 0.3 için tahmin:", est)
assert abs(est - 0.3) <= 1/2**7
print("Alıştırma 4 ✓")

### Alıştırma 5 · Klasik periyot tablosu
`period_table(N)`: gcd(a, N) = 1 olan tüm 1 < a < N için `{a: r}` sözlüğü döndürsün (kaba kuvvetle). N = 15 ve N = 21 için test edilecek.

In [ ]:
def period_table(N):
    # TODO
    pass

t15 = period_table(15)
assert t15 == {2: 4, 4: 2, 7: 4, 8: 4, 11: 2, 13: 4, 14: 2}
t21 = period_table(21)
assert t21[2] == 6 and t21[4] == 3 and t21[5] == 6 and t21[8] == 2
print("N = 21:", t21)
print("Alıştırma 5 ✓")

### Alıştırma 6 · r'den çarpanlara
`factors_from_period(a, r, N)`: r tek ise ya da a^{r/2} ≡ −1 (mod N) ise `None`; aksi hâlde iki çarpanı küçükten büyüğe **tuple** olarak döndürsün.

In [ ]:
def factors_from_period(a, r, N):
    # TODO
    pass

assert factors_from_period(7, 4, 15) == (3, 5)
assert factors_from_period(11, 2, 15) == (3, 5)
assert factors_from_period(14, 2, 15) is None        # a^(r/2) ≡ −1
assert factors_from_period(2, 6, 21) == (3, 7)
assert factors_from_period(4, 3, 21) is None         # r tek
assert factors_from_period(5, 6, 21) is None         # 5³ = 125 ≡ 20 ≡ −1 (mod 21)
print("Alıştırma 6 ✓")

### Alıştırma 7 · Sürekli kesirle r adayı
`period_from_measurement(y, t, N, a)`: `Fraction(y, 2**t).limit_denominator(N)` ile paydayı r₀ alın; a^{r₀} ≢ 1 ise r₀'ın **küçük katlarını** (2r₀, 3r₀, … N'ye kadar) deneyip a^r ≡ 1 sağlayan ilk r'yi döndürün; y = 0 ise `None`.

In [ ]:
def period_from_measurement(y, t, N, a):
    # TODO
    pass

assert period_from_measurement(64, 8, 15, 7) == 4
assert period_from_measurement(192, 8, 15, 7) == 4
assert period_from_measurement(128, 8, 15, 7) == 4      # 1/2 -> 2 -> katı 4
assert period_from_measurement(0, 8, 15, 7) is None
assert period_from_measurement(171, 9, 21, 2) == 6       # 171/512 ≈ 1/3 -> 3 -> katı 6
print("Alıştırma 7 ✓")

### Alıştırma 8 · Uçtan uca Shor (N = 15)
`shor_15(a, shots=64)`: `shor_circuit(a)` devresini çalıştırın, her ölçümden `period_from_measurement` ile r bulun, `factors_from_period` ile çarpanları deneyin; ilk başarılı `(3, 5)` sonucunu döndürün (hiçbiri başarmazsa `None`). a = 2, 7, 13 için test edilecek.

In [ ]:
def shor_15(a, shots=64):
    # TODO
    pass

for a in [2, 7, 13]:
    res = shor_15(a)
    print(f"a = {a}: {res}")
    assert res == (3, 5)
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- **QFT** = durum vektörüne DFT: `QFT(a) = √N · np.fft.ifft(a)` (işaret +, ölçek 1/√N)
- QFT|x⟩ bir **çarpım durumudur**: kübit qⱼ ekvatorda x·2ʲ/N turluk bir **saat ibresi**
- Devre: H + kontrollü faz + SWAP → **O(n²)** kapı (FFT: O(n·2ⁿ)); ama genlikler okunamaz
- **QPE**: kontrollü-U^{2ᵏ} ile faz geri teper, QFT† ibreleri bitlere çevirir → φ ≈ y/2ᵗ
- t sayım kübiti → çözünürlük 1/2ᵗ; tam temsil edilemeyen fazda tepe ≥ 4/π² olasılıkla en yakın değerde
- **Shor** = çarpanlara ayırma → periyot bulma (klasik) + periyot bulma = QPE(modüler çarpma) (kuantum)
- Ölçüm y/2ᵗ ≈ s/r → **sürekli kesir** (`Fraction.limit_denominator`) → r → gcd(a^{r/2} ± 1, N)
- RSA'ya etkisi gerçek ama gelecekte; çözüm bugünden: **FIPS 203/204/205** (ML-KEM, ML-DSA, SLH-DSA)

**Gelecek hafta (Hafta 11):** Varyasyonel algoritmalar — parametreli devreler, klasik optimizasyonla hibrit döngü, parameter-shift kuralı, VQE fikri ve QAOA ile MaxCut. Bugünkü algoritmalar büyük ve hatasız makine isterken, varyasyonel yöntemler bugünün küçük ve gürültülü cihazları için tasarlanmıştır.